# Frozen independent held-out DAS checkpoint

This advisor-facing notebook reads only compact, checksum-locked DAS-only products. It verifies the complete 12-interval freeze, shows how the fixed spatial-support gate reduces 724 base-v1 triggers to 22 DAS-v2 candidates, and provides display-only controls. It never opens network candidates, catalogs, family labels, raw HDF5, or full score arrays. A DAS trigger is an arrival candidate—not yet an earthquake, repeater, or catalog extension.

In [ ]:
from pathlib import Path
import hashlib
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'config' / 'heldout_das_replay.json').is_file()
)
DAS = ROOT / 'outputs' / 'heldout_v2' / 'das'

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

status_path = DAS / 'candidate_generation_status.json'
base_path = DAS / 'base_v1_candidates_time_only.csv'
v2_path = DAS / 'v2_candidates_time_only.csv'
interval_path = DAS / 'interval_status.csv'
status = json.loads(status_path.read_text())
base = pd.read_csv(base_path)
v2 = pd.read_csv(v2_path)
intervals = pd.read_csv(interval_path)

assert status['status'] == 'PASS'
assert status['complete_DAS_candidate_tables_frozen'] is True
assert status['all_12_intervals_PASS'] is True
assert status['all_interval_product_hashes_verified_before_aggregate'] is True
assert len(intervals) == status['interval_count'] == 12
assert intervals['interval_status'].eq('PASS').all()
assert len(base) == status['base_v1_candidate_count'] == 724
assert len(v2) == status['v2_candidate_count'] == 22
assert status['base_v1_candidate_sha256'] == sha256(base_path)
assert status['v2_candidate_sha256'] == sha256(v2_path)
assert status['interval_status_sha256'] == sha256(interval_path)
assert intervals['unique_hdf5_files_read'].sum() == 738
assert status['heldout_DAS_unique_HDF5_files_opened'] == 738
assert status['registered_manifest_file_count'] == 738

access_fields = [
    'network_candidate_time_fields_read',
    'catalog_event_time_fields_read',
    'family_label_fields_read',
]
for table in (base, v2):
    assert table[access_fields].to_numpy().sum() == 0
    assert table['family_assignment'].eq('not_assigned').all()
for field in [
    'network_candidate_table_rows_opened',
    'catalog_association_table_rows_opened',
    'network_or_catalog_candidate_time_fields_read',
    'heldout_family_label_rows_opened',
    'candidate_family_assignments_made',
]:
    assert status[field] == 0

passes_v2 = base['passes_frozen_v2_support_gate'].astype(str).str.lower().eq('true')
assert set(base.loc[passes_v2, 'candidate_id']) == set(v2['parent_v1_candidate_id'])
parents = v2.merge(
    base.add_prefix('base_'),
    left_on='parent_v1_candidate_id',
    right_on='base_candidate_id',
    validate='one_to_one',
)
assert parents['trigger_time'].eq(parents['base_trigger_time']).all()
assert np.allclose(parents['coincidence_score'], parents['base_coincidence_score'])
assert np.array_equal(
    parents['strong_block_count'].to_numpy(),
    parents['base_block_support_count_at_declared_ratio'].to_numpy(),
)
assert not status['threshold_recalibration_or_repair_performed']
assert not status['v2_threshold_or_support_sweep_performed']
assert not status['candidate_deletion_after_review_performed']
print('PASS: complete hashes, 12/12 intervals, exact v2 subset, and zero comparison/label access verify')

In [ ]:
summary = pd.DataFrame({
    'metric': [
        'registered held-out duration (h)',
        'intervals passing fixed mechanics',
        'unique selected HDF5 files read',
        'base-v1 null-threshold triggers retained',
        'DAS-v2 four-of-ten candidates retained',
        'base-v1 triggers rejected by v2 gate',
        'intervals containing a v2 candidate',
        'network/catalog candidate-time fields read',
        'family assignments made',
    ],
    'value': [
        status['heldout_total_duration_h'],
        int(intervals['interval_status'].eq('PASS').sum()),
        status['heldout_DAS_unique_HDF5_files_opened'],
        len(base),
        len(v2),
        status['v2_rejected_base_candidate_count'],
        int(v2['interval_id'].nunique()),
        status['network_or_catalog_candidate_time_fields_read'],
        status['candidate_family_assignments_made'],
    ],
})
display(summary)
print('Scientific extension gate:', status['scientific_extension_claim_gate'])
print('Next gate:', status['next_stage_gate'])

In [ ]:
per_interval = intervals.copy()
per_interval['v2_retention_percent'] = (
    100 * per_interval['v2_candidate_count'] / per_interval['base_v1_candidate_count']
)
per_interval['maximum_over_null_threshold'] = (
    per_interval['observed_maximum_score'] / per_interval['detection_threshold']
)
display(per_interval[[
    'interval_id', 'interval_status', 'selected_manifest_record_count',
    'usable_sampled_channel_count', 'usable_block_count',
    'detection_threshold', 'observed_maximum_score',
    'base_v1_candidate_count', 'v2_candidate_count',
    'v2_retention_percent', 'maximum_over_null_threshold',
]])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x = np.arange(len(per_interval))
width = 0.38
axes[0].bar(x - width / 2, per_interval['base_v1_candidate_count'], width, label='base v1')
axes[0].bar(x + width / 2, per_interval['v2_candidate_count'], width, label='DAS v2')
axes[0].set_ylabel('Frozen candidate count')
axes[0].set_title('Fixed spatial-support filtering')
axes[0].legend()
axes[1].plot(x, per_interval['detection_threshold'], 'o-', label='95% block-shift null threshold')
axes[1].plot(x, per_interval['observed_maximum_score'], 'o-', label='observed maximum')
axes[1].set_ylabel('Coincidence score')
axes[1].set_title('Per-interval threshold and maximum')
axes[1].legend(fontsize=8)
for ax in axes:
    ax.set_xticks(x, per_interval['interval_id'], rotation=60)
    ax.grid(axis='y', alpha=0.25)
fig.tight_layout()
plt.show()

In [ ]:
timing = v2.merge(
    intervals[['interval_id', 'interval_start_utc']],
    on='interval_id',
    validate='many_to_one',
).copy()
timing['minute_in_interval'] = (
    pd.to_datetime(timing['trigger_time'], utc=True)
    - pd.to_datetime(timing['interval_start_utc'], utc=True)
).dt.total_seconds() / 60
timing['score_over_null_threshold'] = (
    timing['coincidence_score'] / timing['v1_interval_null_threshold']
)
active_intervals = list(timing['interval_id'].drop_duplicates())
y_lookup = {interval_id: index for index, interval_id in enumerate(active_intervals)}
fig, ax = plt.subplots(figsize=(10, 3.8))
points = ax.scatter(
    timing['minute_in_interval'],
    timing['interval_id'].map(y_lookup),
    s=25 * timing['strong_block_count'],
    c=timing['score_over_null_threshold'],
    cmap='viridis',
    edgecolor='black',
    linewidth=0.4,
)
ax.set_yticks(range(len(active_intervals)), active_intervals)
ax.set_xlim(0, 60)
ax.set_xlabel('Trigger minute within registered hour')
ax.set_title('Frozen DAS-v2 arrival candidates (size = strong-block count)')
ax.grid(alpha=0.25)
fig.colorbar(points, ax=ax, label='score / interval null threshold')
fig.tight_layout()
plt.show()

In [ ]:
# Display-only advisor controls. These cannot rewrite or reclassify a frozen product.
DISPLAY_INTERVAL = 'all'       # e.g. 'heldout_04'
DISPLAY_MIN_STRONG_BLOCKS = 4  # exploratory display threshold, 4--10
DISPLAY_MIN_SCORE_RATIO = 1.0  # coincidence score / registered null threshold
MAX_ROWS = 30

sandbox = timing.copy()
if DISPLAY_INTERVAL != 'all':
    sandbox = sandbox[sandbox['interval_id'] == DISPLAY_INTERVAL]
sandbox = sandbox[
    (sandbox['strong_block_count'] >= DISPLAY_MIN_STRONG_BLOCKS)
    & (sandbox['score_over_null_threshold'] >= DISPLAY_MIN_SCORE_RATIO)
]
display(sandbox.groupby('interval_id').size().rename('displayed_candidates').to_frame())
display(sandbox.sort_values(
    ['interval_id', 'trigger_epoch_s']
).head(MAX_ROWS)[[
    'interval_id', 'candidate_id', 'trigger_time', 'minute_in_interval',
    'coincidence_score', 'v1_interval_null_threshold',
    'score_over_null_threshold', 'strong_block_count',
]])
print('Display-only result: the registered four-block candidate set remains unchanged on disk.')

## Decision and next gate

The independent held-out DAS computation is technically successful and yields a nonempty, auditable candidate population: 22 of 724 base triggers survive the frozen four-of-ten rule, concentrated in four of twelve hours. That concentration—including 13 triggers in one hour—and the single 10/10 high-score trigger are scientifically interesting but ambiguous before comparison. They could include local earthquakes, regional arrivals, repeated cultural/instrumental signals, or detector artifacts.

The next allowed operation is a separately committed time-only DAS-versus-network comparison that pins these hashes and its one-to-one timing rule before opening either candidate-time table together. Catalog and family-label adjudication remain later, separately registered stages. Until those gates pass, this checkpoint demonstrates successful independent DAS candidate generation—not DAS incremental value, a repeater-family extension, stress drop, or creep rate.